In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

llm = ChatGroq(model="llama-3.3-70b-versatile")

In [3]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [4]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [5]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [7]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.',
 'explanation': 'A classic play on words. The joke "Why was the pizza in a bad mood? Because it was feeling a little crusty" is a pun that relies on a double meaning of the word "crusty."\n\nIn one sense, "crusty" refers to the outer layer of a pizza, which is typically crispy and golden brown. However, "crusty" can also be used to describe someone\'s personality or mood, implying that they are irritable, grumpy, or disagreeable.\n\nThe joke sets up the expectation that the pizza\'s bad mood will be explained by some typical reason, such as being overcooked or having too many toppings. But instead, the punchline subverts this expectation by using the word "crusty" in a way that references both the pizza\'s crust and its emotional state.\n\nThe humor comes from the unexpected twist on the word\'s meaning, creating a clever and silly connection between the setup and the punchline. I

In [8]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'A classic play on words. The joke "Why was the pizza in a bad mood? Because it was feeling a little crusty" is a pun that relies on a double meaning of the word "crusty."\n\nIn one sense, "crusty" refers to the outer layer of a pizza, which is typically crispy and golden brown. However, "crusty" can also be used to describe someone\'s personality or mood, implying that they are irritable, grumpy, or disagreeable.\n\nThe joke sets up the expectation that the pizza\'s bad mood will be explained by some typical reason, such as being overcooked or having too many toppings. But instead, the punchline subverts this expectation by using the word "crusty" in a way that references both the pizza\'s crust and its emotional state.\n\nThe humor comes from the unexpected twist on the word\'s meaning, creating a clever and silly connection between the setup a

In [9]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'A classic play on words. The joke "Why was the pizza in a bad mood? Because it was feeling a little crusty" is a pun that relies on a double meaning of the word "crusty."\n\nIn one sense, "crusty" refers to the outer layer of a pizza, which is typically crispy and golden brown. However, "crusty" can also be used to describe someone\'s personality or mood, implying that they are irritable, grumpy, or disagreeable.\n\nThe joke sets up the expectation that the pizza\'s bad mood will be explained by some typical reason, such as being overcooked or having too many toppings. But instead, the punchline subverts this expectation by using the word "crusty" in a way that references both the pizza\'s crust and its emotional state.\n\nThe humor comes from the unexpected twist on the word\'s meaning, creating a clever and silly connection between the setup 

In [10]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a relationship.',
 'explanation': 'A clever play on words. This joke is funny because it uses a common phrase associated with romantic relationships, "tangled up," and gives it a literal twist. \n\nIn relationships, "tangled up" typically means to become deeply emotionally involved or complicated. However, in this joke, the phrase is applied to spaghetti, which is a type of long, thin, and flexible noodle that can easily become tangled or knotted. \n\nThe punchline "afraid of getting tangled up in a relationship" is humorous because it takes the usual meaning of the phrase and replaces it with a literal interpretation that is relevant to the physical properties of spaghetti. The joke relies on this wordplay to create a clever and amusing connection between the setup and the punchline, making it a lighthearted and entertaining joke.'}

In [12]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a relationship.', 'explanation': 'A clever play on words. This joke is funny because it uses a common phrase associated with romantic relationships, "tangled up," and gives it a literal twist. \n\nIn relationships, "tangled up" typically means to become deeply emotionally involved or complicated. However, in this joke, the phrase is applied to spaghetti, which is a type of long, thin, and flexible noodle that can easily become tangled or knotted. \n\nThe punchline "afraid of getting tangled up in a relationship" is humorous because it takes the usual meaning of the phrase and replaces it with a literal interpretation that is relevant to the physical properties of spaghetti. The joke relies on this wordplay to create a clever and amusing connection between the setup and the punchline, making it a lighthearted and entertaining joke.'}, next=(), c

In [13]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'A classic play on words. The joke "Why was the pizza in a bad mood? Because it was feeling a little crusty" is a pun that relies on a double meaning of the word "crusty."\n\nIn one sense, "crusty" refers to the outer layer of a pizza, which is typically crispy and golden brown. However, "crusty" can also be used to describe someone\'s personality or mood, implying that they are irritable, grumpy, or disagreeable.\n\nThe joke sets up the expectation that the pizza\'s bad mood will be explained by some typical reason, such as being overcooked or having too many toppings. But instead, the punchline subverts this expectation by using the word "crusty" in a way that references both the pizza\'s crust and its emotional state.\n\nThe humor comes from the unexpected twist on the word\'s meaning, creating a clever and silly connection between the setup 

### Time Travel

In [14]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [15]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'A classic play on words. The joke "Why was the pizza in a bad mood? Because it was feeling a little crusty" is a pun that relies on a double meaning of the word "crusty."\n\nIn one sense, "crusty" refers to the outer layer of a pizza, which is typically crispy and golden brown. However, "crusty" can also be used to describe someone\'s personality or mood, implying that they are irritable, grumpy, or disagreeable.\n\nThe joke sets up the expectation that the pizza\'s bad mood will be explained by some typical reason, such as being overcooked or having too many toppings. But instead, the punchline subverts this expectation by using the word "crusty" in a way that references both the pizza\'s crust and its emotional state.\n\nThe humor comes from the unexpected twist on the word\'s meaning, creating a clever and silly connection between the setup 

### Fault Tolerance

In [16]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [17]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [18]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [19]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")